In [1]:
import os
PROJECT = "/Users/alice/Projects/nn-optimization-project"
os.chdir(PROJECT)
print(os.getcwd())

/Users/alice/Projects/nn-optimization-project


In [2]:
import torchvision as tv
import matplotlib.pyplot as plt

In [ ]:
train_full = tv.datasets.CIFAR10(root="data", train=True, download=True)
test = tv.datasets.CIFAR10(root="data", train=False, download=True)

In [ ]:
print(len(train_full),
len(test),
len(train_full.classes))

In [ ]:
fig, axes = plt.subplots(4,4, figsize=(8, 8))
for i in range(16):
    img, label = train_full[i]
    ax = axes.flat[i]
    ax.imshow(img)
    ax.set_title(train_full.classes[label])

plt.tight_layout()
plt.savefig("figures/cifar_examples.png", dpi=150)
plt.show()

In [ ]:
import collections
counts = collections.Counter(train_full.targets)

for label, n in sorted(counts.items()):
    print(f"{label} {train_full.classes[label]:<12} {n}")

***Phase 2***

In [ ]:
data = train_full.data / 255
print(data.shape)   # (50000, 32, 32, 3)

In [ ]:
mean = data.mean(axis=(0, 1, 2))
std = data.std(axis=(0, 1, 2))
print(mean, std)

In [ ]:
CIFAR_MEAN = (0.49139968, 0.48215841, 0.44653091)
CIFAR_STD = (0.24703223, 0.24348513, 0.26158784)

In [ ]:
transform = tv.transforms.Compose([
    tv.transforms.ToTensor(),
    tv.transforms.Normalize(mean=CIFAR_MEAN, std=CIFAR_STD),
])
ds = tv.datasets.CIFAR10("data", train=True, download=True,transform=transform)
x, y = ds[0]
print(x.shape, x.dtype, x.min().item(), x.max().item())

In [ ]:
from src.data import get_transforms

train_tf = get_transforms(train=True)
eval_tf = get_transforms(train=False)
print(train_tf)
print(eval_tf)

***PHASE 3***

In [ ]:
import torch
from torch.utils.data import Subset
g = torch.Generator().manual_seed(42)
perm = torch.randperm(50000, generator=g).tolist()

In [ ]:
train_aug = tv.datasets.CIFAR10(root="data", train=True, transform=get_transforms(train=True))
train_eval = tv.datasets.CIFAR10(root="data", train=True, transform=get_transforms(train=False))

In [ ]:
train_idx = perm[:45_000]
val_idx = perm[45_000:]

In [ ]:
train_set = Subset(train_aug, train_idx)   # with augmentering
val_set = Subset(train_eval, val_idx)      # without augmentering

In [ ]:
print(len(train_set), len(val_set))          # 45000 5000
print(len(set(train_idx) & set(val_idx)))    # 0, no image in both
print(val_idx[:5])                           # same 5 numbers eveytime we run

***PHASE 4***

In [ ]:
from src.data import get_loaders

In [ ]:
train_loader, val_loader, test_loader = get_loaders(128)
print(len(train_loader), len(val_loader), len(test_loader))   # 352 40 79

***PHASE 5***

In [ ]:
from src.models import SimpleCNN
model = SimpleCNN()
x = torch.randn(8, 3, 32, 32)
print(model.features(x).shape)   # torch.Size([8, 128, 4, 4])

In [ ]:
model = SimpleCNN()
x = torch.randn(8, 3, 32, 32)
print(model(x).shape)                               # torch.Size([8, 10])
print(sum(p.numel() for p in model.parameters()))   # 620810

In [ ]:
from src.models import build_model
model = build_model("simple_cnn")
print(model(torch.randn(8, 3, 32, 32)).shape)   # torch.Size([8, 10])
build_model("fel_namn")                          # ska ge ValueError